In [5]:
import pyspark
print(f"Sua versão do PySpark é: {pyspark.__version__}")

Sua versão do PySpark é: 3.5.1


In [6]:
from pyspark.sql import SparkSession

# Configuração otimizada para PySpark 3.5.1
spark = SparkSession.builder \
    .appName("IcebergFootball") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "../data/iceberg") \
    .getOrCreate()

print(f"Sessão iniciada com Sucesso! Versão do Spark: {spark.version}")

Sessão iniciada com Sucesso! Versão do Spark: 3.5.1


In [7]:
# 1. Criar a tabela fato_evento
spark.sql("""
    CREATE TABLE IF NOT EXISTS local.fato_evento (
        id_evento INT,
        id_partida INT,
        id_jogador INT,
        tipo_evento STRING,
        minuto_jogo INT,
        revisado_var BOOLEAN
    ) USING iceberg
""")

# 2. Inserir dados iniciais (Flamengo x Vasco)
spark.sql("""
    INSERT INTO local.fato_evento VALUES 
    (1, 101, 10, 'Gol', 15, false),
    (2, 101, 5, 'Cartão Amarelo', 30, false),
    (3, 101, 9, 'Gol', 44, false)
""")

print("Tabela inicial no Apache Iceberg:")
spark.sql("SELECT * FROM local.fato_evento").orderBy("id_evento").show()

Tabela inicial no Apache Iceberg:
+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        10|           Gol|         15|       false|
|        2|       101|         5|Cartão Amarelo|         30|       false|
|        3|       101|         9|           Gol|         44|       false|
+---------+----------+----------+--------------+-----------+------------+



In [8]:
# 1. Corrigindo o autor do gol (id_evento 1)
spark.sql("""
    UPDATE local.fato_evento 
    SET id_jogador = 99, revisado_var = true 
    WHERE id_evento = 1
""")

# 2. Anulando o gol por impedimento (id_evento 3)
spark.sql("DELETE FROM local.fato_evento WHERE id_evento = 3")

print("Placar Final atualizado (Apache Iceberg):")
spark.sql("SELECT * FROM local.fato_evento").orderBy("id_evento").show()

Placar Final atualizado (Apache Iceberg):
+---------+----------+----------+--------------+-----------+------------+
|id_evento|id_partida|id_jogador|   tipo_evento|minuto_jogo|revisado_var|
+---------+----------+----------+--------------+-----------+------------+
|        1|       101|        99|           Gol|         15|        true|
|        2|       101|         5|Cartão Amarelo|         30|       false|
+---------+----------+----------+--------------+-----------+------------+

